# 🚀 Agent 工具模式与最佳实践

**欢迎来到 Kaggle 5 天 Agents 课程 Day 2！**

在上一份 Notebook 中，你学会了如何把自定义 Python 函数作为工具接入 Agent。本 Notebook 会进一步深入：**接入外部 MCP 服务**，并处理**长时间运行操作（Long-Running Operations）**。

在本 Notebook 中，你将学习如何：

- ✅ **连接外部 MCP 服务器**
- ✅ 实现可暂停的**长时间运行操作**（等待外部输入）
- ✅ 构建可恢复的工作流，在对话中断后继续保持状态
- ✅ 理解这些模式的适用时机与使用方式

---
## ⚙️ 第 1 部分：环境准备

### 1.1：安装依赖

Kaggle Notebooks 环境已预装 Python 版 [google-adk](https://google.github.io/adk-docs/) 及其依赖。

如果你要在课程外、自己的 Python 开发环境中安装并使用 ADK，可运行：

```
pip install google-adk
```

### 1.2：配置 Gemini API Key

本 Notebook 使用 [Gemini API](https://ai.google.dev/gemini-api/)，需要 API key。

**1. 获取 API key**

如果你还没有，可在 [Google AI Studio 创建 API key](https://aistudio.google.com/app/api-keys)。

**2. 将 key 添加到 Kaggle Secrets**

接下来，你需要把 API key 作为 Kaggle User Secret 添加到 Notebook。

1. 在 Notebook 编辑器顶部菜单选择 `Add-ons` -> `Secrets`。
2. 新建标签为 `GOOGLE_API_KEY` 的 secret。
3. 把 API key 粘贴到 "Value" 字段并点击 "Save"。
4. 确认 `GOOGLE_API_KEY` 左侧复选框已勾选，使其附加到该 Notebook。

**3. 在 Notebook 中鉴权**

运行下面代码单元，读取你保存的 `GOOGLE_API_KEY` 并设置为环境变量：

In [1]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Setup and authentication complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Setup and authentication complete.


### 1.3：导入 ADK 组件

现在从 Agent Development Kit 中导入你将用到的组件。这样可以让代码结构更清晰，也确保我们具备必要构建块。

In [2]:
import uuid
from google.genai import types

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.tool_context import ToolContext
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

from google.adk.apps.app import App, ResumabilityConfig
from google.adk.tools.function_tool import FunctionTool

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


### 1.4：配置重试选项

在使用 LLM 时，你可能会遇到速率限制或服务临时不可用等瞬时错误。重试选项会通过指数退避自动重试请求。

In [4]:
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

---
## 🧰 第 2 部分：Model Context Protocol

到目前为止，你已经学习了如何为 Agent 编写自定义函数工具。但要连接外部系统（GitHub、数据库、Slack），通常需要自己编写并维护 API 客户端。

**Model Context Protocol（MCP）** 是一个开放标准，让 Agent 可以直接复用社区构建的集成能力。你无需重复造轮子，连接现有 MCP server 即可。

MCP 让 Agent 能够：

✅ **访问实时外部数据**（数据库、API、服务），而无需写自定义集成代码  
✅ **复用社区工具生态**，通过统一标准接口调用  
✅ **按需扩展能力**，连接多个专用服务器

### 2.1：MCP 是如何工作的

MCP 把你的 Agent（**client**）连接到提供工具的外部 **MCP servers**：

- **MCP Server**：提供具体工具（如图像生成、数据库访问）
- **MCP Client**：你的 Agent，负责调用这些工具
- **统一接口**：不同服务器遵循同一标准调用方式

**架构：**
```
┌──────────────────┐
│   Your Agent     │
│   (MCP Client)   │
└────────┬─────────┘
         │
         │ Standard MCP Protocol
         │
    ┌────┴────┬────────┬────────┐
    │         │        │        │
    ▼         ▼        ▼        ▼
┌────────┐ ┌─────┐ ┌──────┐ ┌─────┐
│ GitHub │ │Slack│ │ Maps │ │ ... │
│ Server │ │ MCP │ │ MCP  │ │     │
└────────┘ └─────┘ └──────┘ └─────┘
```

### 2.2：在 Agent 中使用 MCP

流程很简单：

1. 选择一个 MCP Server 及其工具
2. 创建 MCP Toolset（配置连接）
3. 把它加入 Agent
4. 运行并测试 Agent

**步骤 1：选择 MCP Server**

本示例使用 **[Everything MCP Server](https://github.com/modelcontextprotocol/servers/tree/main/src/everything)**，它是一个 npm 包（`@modelcontextprotocol/server-everything`），专用于 MCP 集成测试。

它提供 `getTinyImage` 工具，返回一个简单测试图片（16x16 像素，Base64 编码）。**更多服务器可见：** [modelcontextprotocol.io/examples](https://modelcontextprotocol.io/examples)

**‼️ 注意：这是教学演示服务器。** 在生产场景中，你会接入 Google Maps、Slack、Discord 等实际服务。

**步骤 2：创建 MCP Toolset**

`McpToolset` 用于把 ADK Agent 与 MCP Server 连接起来。

**这段代码做了什么：**
- 使用 `npx`（Node 包运行器）启动 MCP server
- 连接到 `@modelcontextprotocol/server-everything`
- 通过过滤只启用 `getTinyImage`（服务器还有其他工具，这里只用它）

In [5]:
# MCP integration with Everything Server
mcp_image_server = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="npx",  # Run MCP server via npx
            args=[
                "-y",  # Argument for npx to auto-confirm install
                "@modelcontextprotocol/server-everything",
            ],
            tool_filter=["getTinyImage"],
        ),
        timeout=30,
    )
)

print("✅ MCP Tool created")

✅ MCP Tool created


#### **幕后过程：**
1. **启动服务**：ADK 运行 `npx -y @modelcontextprotocol/server-everything`
2. **握手连接**：建立 stdio 通信通道
3. **工具发现**：服务端告知 ADK："我提供 getTinyImage 功能"
4. **自动集成**：工具自动出现在 Agent 的工具列表中
5. **执行调用**：当 Agent 调用 `getTinyImage()` 时，ADK 转发给 MCP server
6. **返回结果**：server 结果无缝返回给 Agent

**为什么重要：** 无需手写集成代码，就能立即获得工具能力。

**步骤 3：把 MCP 工具加入 Agent**

把 `mcp_server` 加入 Agent 的 tools 列表，并更新 instruction 以处理“生成微型图片”的请求。

In [6]:
# Create image agent with MCP integration
image_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="image_agent",
    instruction="Use the MCP Tool to generate images for user queries",
    tools=[mcp_image_server],
)

创建 runner：

In [7]:
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(agent=image_agent)

**步骤 4：测试 Agent**

让 Agent 生成一张图，观察它如何调用 MCP 工具：

In [8]:
response = await runner.run_debug("Provide a sample tiny image", verbose=True)


 ### Created new session: debug_session_id

User > Provide a sample tiny image


/usr/local/lib/python3.11/dist-packages/google/adk/tools/mcp_tool/mcp_tool.py:101: UserWarning: [EXPERIMENTAL] BaseAuthenticatedTool: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__(


image_agent > [Calling tool: getTinyImage({})]
image_agent > [Tool result: {'content': [{'type': 'text', 'text': 'This is a tiny image:'}, {'type': 'image', 'data': 'iVBORw0KG...]
image_agent > This is a tiny image:

The image above is the MCP tiny image.



**显示图片：**

服务端返回的是 base64 编码图片数据。我们将其解码并展示：

In [9]:
from IPython.display import display, Image as IPImage
import base64

for event in response:
    if event.content and event.content.parts:
        for part in event.content.parts:
            if hasattr(part, "function_response") and part.function_response:
                for item in part.function_response.response.get("content", []):
                    if item.get("type") == "image":
                        display(IPImage(data=base64.b64decode(item["data"])))

### 2.3：扩展到其他 MCP Servers

同样的模式适用于任意 MCP server，变化点仅在 `connection_params`。下面给出几个示例：

#### **👉 Kaggle MCP Server** - 用于数据集与 Notebook 操作

Kaggle 提供了 MCP server，让 Agent 能与 Kaggle 数据集、Notebook、竞赛信息交互。

**连接示例：**
```python
McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command='npx',
            args=[
                '-y',
                'mcp-remote',
                'https://www.kaggle.com/mcp'
            ],
        ),
        timeout=30,
    )
)
```

**可提供能力：**
- 📊 搜索与下载 Kaggle 数据集
- 📓 访问 Notebook 元数据
- 🏆 查询竞赛信息等

**更多信息：** [Kaggle MCP Documentation](https://www.kaggle.com/docs/mcp)

#### **👉 GitHub MCP Server** - 用于 PR/Issue 分析

```python
McpToolset(
    connection_params=StreamableHTTPServerParams(
        url="https://api.githubcopilot.com/mcp/",
        headers={
            "Authorization": f"Bearer {GITHUB_TOKEN}",
            "X-MCP-Toolsets": "all",
            "X-MCP-Readonly": "true"
        },
    ),
)
```

**更多资源：** [ADK Third-party Tools Documentation](https://google.github.io/adk-docs/tools/third-party/)

---
## 🔄 第 3 部分：长时间运行操作（Human-in-the-Loop）

到目前为止，我们所有工具都是“调用后立即返回”：


```User asks → Agent calls tool → Tool returns result → Agent responds```


**但如果工具执行很久，或你需要人在执行前审批怎么办？**

示例：发货 Agent 在创建大额订单前应该先请求审批。


```User asks → Agent calls tool → Tool PAUSES and asks human → Human approves → Tool completes → Agent responds```


这就是 **Long-Running Operation（LRO）**：工具需要先暂停，等待外部输入（人工审批），再恢复执行。

**何时使用 Long-Running Operations：**

- 💰 **金融交易审批**（转账、采购）
- 🗑️ **批量操作**（比如删除 1000 条记录前先确认）
- 📋 **合规检查点**（监管审批）
- 💸 **高成本动作**（例如一次启动 50 台服务器）
- ⚠️ **不可逆操作**（永久删除账号）

### 3.1：今天要构建什么

我们将构建一个**发货协调 Agent（包含一个工具）**，它可以：
- 小订单自动通过（≤5 个集装箱）
- 大订单（>5 个）时**暂停并请求审批**
- 根据审批结果完成或取消订单

这个示例体现了长时间运行操作的核心模式：**暂停 → 等待人工输入 → 恢复**。

### 3.2：带审批逻辑的发货工具

下面是完整函数。

#### `ToolContext` 参数

注意函数签名里包含 `tool_context: ToolContext`。当工具运行时，ADK 会自动注入该对象。它提供两项关键能力：

1. **请求审批**：调用 `tool_context.request_confirmation()`
2. **读取审批状态**：访问 `tool_context.tool_confirmation`

In [11]:
LARGE_ORDER_THRESHOLD = 5


def place_shipping_order(
    num_containers: int, destination: str, tool_context: ToolContext
) -> dict:
    """Places a shipping order. Requires approval if ordering more than 5 containers (LARGE_ORDER_THRESHOLD).

    Args:
        num_containers: Number of containers to ship
        destination: Shipping destination

    Returns:
        Dictionary with order status
    """

    # -----------------------------------------------------------------------------------------------
    # -----------------------------------------------------------------------------------------------
    # SCENARIO 1: Small orders (≤5 containers) auto-approve
    if num_containers <= LARGE_ORDER_THRESHOLD:
        return {
            "status": "approved",
            "order_id": f"ORD-{num_containers}-AUTO",
            "num_containers": num_containers,
            "destination": destination,
            "message": f"Order auto-approved: {num_containers} containers to {destination}",
        }

    # -----------------------------------------------------------------------------------------------
    # -----------------------------------------------------------------------------------------------
    # SCENARIO 2: This is the first time this tool is called. Large orders need human approval - PAUSE here.
    if not tool_context.tool_confirmation:
        tool_context.request_confirmation(
            hint=f"⚠️ Large order: {num_containers} containers to {destination}. Do you want to approve?",
            payload={"num_containers": num_containers, "destination": destination},
        )
        return {  # This is sent to the Agent
            "status": "pending",
            "message": f"Order for {num_containers} containers requires approval",
        }

    # -----------------------------------------------------------------------------------------------
    # -----------------------------------------------------------------------------------------------
    # SCENARIO 3: The tool is called AGAIN and is now resuming. Handle approval response - RESUME here.
    if tool_context.tool_confirmation.confirmed:
        return {
            "status": "approved",
            "order_id": f"ORD-{num_containers}-HUMAN",
            "num_containers": num_containers,
            "destination": destination,
            "message": f"Order approved: {num_containers} containers to {destination}",
        }
    else:
        return {
            "status": "rejected",
            "message": f"Order rejected: {num_containers} containers to {destination}",
        }


print("✅ Long-running functions created!")

✅ Long-running functions created!


### 3.3：理解这段代码

你已经看过完整函数，下面拆解其工作机制。

<img src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day2/lro-tool.png" width="1000" alt="Long-running operation tool">

#### 三种场景如何工作

该工具通过检查 `tool_context.tool_confirmation` 处理三类场景：

**场景 1：小订单（≤5 个）**：立即返回自动通过状态。
- 不会触发 `tool_context.tool_confirmation` 检查

**场景 2：大订单 - 首次调用**
- 工具识别为首次调用：`if not tool_context.tool_confirmation:`
- 调用 `request_confirmation()` 请求人工审批
- 立即返回 `{'status': 'pending', ...}`
- **ADK 自动生成 `adk_request_confirmation` 事件**
- Agent 执行暂停，等待人工决策

**场景 3：大订单 - 恢复调用**
- 工具识别为恢复执行：`if not tool_context.tool_confirmation:` 为 False
- 检查人工决策：`tool_context.tool_confirmation.confirmed`
- 若 True -> 返回 approved
- 若 False -> 返回 rejected

**关键理解：** 两次调用之间，需要由你的工作流代码（Section 4）检测 `adk_request_confirmation` 事件，并携带审批结果恢复执行。

### 3.4：创建 Agent、App 与 Runner

**步骤 1：创建 Agent**

把工具加入 Agent。该工具会根据订单规模在内部决定是否请求审批。

In [12]:
# Create shipping agent with pausable tool
shipping_agent = LlmAgent(
    name="shipping_agent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""You are a shipping coordinator assistant.
  
  When users request to ship containers:
   1. Use the place_shipping_order tool with the number of containers and destination
   2. If the order status is 'pending', inform the user that approval is required
   3. After receiving the final result, provide a clear summary including:
      - Order status (approved/rejected)
      - Order ID (if available)
      - Number of containers and destination
   4. Keep responses concise but informative
  """,
    tools=[FunctionTool(func=place_shipping_order)],
)

print("✅ Shipping Agent created!")

✅ Shipping Agent created!


**步骤 2：封装为可恢复 App**

**问题：** 普通 `LlmAgent` 是无状态的，每次调用相互独立，不会记住之前发生了什么。如果工具请求审批，Agent 无法记住自己停在了哪一步。

**解决方案：** 把 Agent 封装进启用可恢复能力的 **`App`**。它会增加持久化层来保存并恢复状态。

**工具暂停时会保存什么：**
- 当前为止的全部对话消息
- 调用了哪个工具（`place_shipping_order`）
- 工具参数（例如 10 个集装箱，Rotterdam）
- 精确暂停点（正在等待审批）

恢复时，App 会加载这些状态，让 Agent 从中断点继续执行，就像时间没有流逝。

In [13]:
# Wrap the agent in a resumable app - THIS IS THE KEY FOR LONG-RUNNING OPERATIONS!
shipping_app = App(
    name="shipping_coordinator",
    root_agent=shipping_agent,
    resumability_config=ResumabilityConfig(is_resumable=True),
)

print("✅ Resumable app created!")

✅ Resumable app created!


/tmp/ipykernel_47/3673777575.py:5: UserWarning: [EXPERIMENTAL] ResumabilityConfig: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  resumability_config=ResumabilityConfig(is_resumable=True),


**步骤 3：基于 App 创建 Session 与 Runner**

通过 `app=shipping_app`（而不是 `agent=...`）传入 runner，让它具备可恢复能力。

In [14]:
session_service = InMemorySessionService()

# Create runner with the resumable app
shipping_runner = Runner(
    app=shipping_app,  # Pass the app instead of the agent
    session_service=session_service,
)

print("✅ Runner created!")

✅ Runner created!


---

**✅ 小结：你的可暂停发货 Agent 已经搭建完成！**

你已经创建了：
1. ✅ 可暂停等待审批的工具（`place_shipping_order`）
2. ✅ 使用该工具的 Agent（`shipping_agent`）
3. ✅ 负责保存状态的可恢复 App（`shipping_app`）
4. ✅ 支持暂停/恢复的 Runner（`shipping_runner`）

**下一步：** 编写工作流代码，测试 Agent 如何检测暂停并处理审批。

---

## 🏗️ 第 4 部分：构建工作流

‼️ **重要：** 这部分工作流代码会用到 ADK 的 Session、Runner、Event 等概念。本 Notebook 会覆盖你完成长时间运行操作所需的关键内容。更深入的原理会在 Day 3 详细讲解，你也可参考 [ADK docs](https://google.github.io/adk-docs/runtime/) 与这个 [视频](https://www.youtube.com/watch?v=44C8u0CDtSo&list=PLOU2XLYxmsIIAPgM8FmtEcFTXLLzmh4DK&index=2&t=1s)。

### 4.1：⚠️ 关键点 - 在工作流里处理事件

Agent 不会自动完成暂停/恢复。**每个长时间运行工作流都需要你显式实现：**

1. **检测暂停点**：检查事件里是否存在 `adk_request_confirmation`
2. **获取人工决策**：生产环境通常由 UI 等待用户点击，这里我们用模拟方式
3. **恢复执行**：带上保存的 `invocation_id` 把决策发送回 Agent

### 4.2 理解关键技术概念

👉 **`events`** - Agent 执行过程中，ADK 会持续生成事件：工具调用、模型响应、函数结果都会变成事件

👉 **`adk_request_confirmation` 事件** - 这是一个特殊事件，表示“这里需要暂停”
- 当工具调用 `request_confirmation()` 时由 ADK 自动生成
- 事件中包含 `invocation_id`
- 你的工作流需要捕获它，才能知道 Agent 已暂停

👉 **`invocation_id`** - 每次 `run_async()` 调用都会有唯一 ID（如 "abc123"）
- 当工具暂停时，你要保存这个 ID
- 恢复时必须传回同一个 ID，让 ADK 知道该续跑哪次执行
- 若不用同一个 ID，ADK 会新开一次执行而不是恢复

### 4.3：用于处理事件的辅助函数

这些函数用于封装事件遍历逻辑。

**`check_for_approval()`** - 检测 Agent 是否已暂停
- 遍历所有事件并查找特殊事件 `adk_request_confirmation`
- 返回 `approval_id`（本次审批请求标识）和 `invocation_id`（恢复执行标识）
- 若未检测到暂停则返回 `None`

In [15]:
def check_for_approval(events):
    """Check if events contain an approval request.

    Returns:
        dict with approval details or None
    """
    for event in events:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if (
                    part.function_call
                    and part.function_call.name == "adk_request_confirmation"
                ):
                    return {
                        "approval_id": part.function_call.id,
                        "invocation_id": event.invocation_id,
                    }
    return None

**`print_agent_response()`** - 打印 Agent 文本响应
- 简单辅助函数：从事件中提取并输出文本

In [16]:
def print_agent_response(events):
    """Print agent's text responses from events."""
    for event in events:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f"Agent > {part.text}")

**`create_approval_response()`** - 组织人工审批结果
- 接收审批信息与人工布尔决策（True/False）
- 构造 ADK 可识别的 `FunctionResponse`
- 再包装为 `Content` 对象发送给 Agent

In [17]:
def create_approval_response(approval_info, approved):
    """Create approval response message."""
    confirmation_response = types.FunctionResponse(
        id=approval_info["approval_id"],
        name="adk_request_confirmation",
        response={"confirmed": approved},
    )
    return types.Content(
        role="user", parts=[types.Part(function_response=confirmation_response)]
    )


print("✅ Helper functions defined")

✅ Helper functions defined


### 4.4：工作流函数 - 把这些拼起来！

`run_shipping_workflow()` 负责编排完整审批流程。

具体代码解读见下方单元。

<img src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day2/lro-workflow.png" width="1000" alt="Long-running operation workflow">

In [18]:
async def run_shipping_workflow(query: str, auto_approve: bool = True):
    """Runs a shipping workflow with approval handling.

    Args:
        query: User's shipping request
        auto_approve: Whether to auto-approve large orders (simulates human decision)
    """

    print(f"\n{'='*60}")
    print(f"User > {query}\n")

    # Generate unique session ID
    session_id = f"order_{uuid.uuid4().hex[:8]}"

    # Create session
    await session_service.create_session(
        app_name="shipping_coordinator", user_id="test_user", session_id=session_id
    )

    query_content = types.Content(role="user", parts=[types.Part(text=query)])
    events = []

    # -----------------------------------------------------------------------------------------------
    # -----------------------------------------------------------------------------------------------
    # STEP 1: Send initial request to the Agent. If num_containers > 5, the Agent returns the special `adk_request_confirmation` event
    async for event in shipping_runner.run_async(
        user_id="test_user", session_id=session_id, new_message=query_content
    ):
        events.append(event)

    # -----------------------------------------------------------------------------------------------
    # -----------------------------------------------------------------------------------------------
    # STEP 2: Loop through all the events generated and check if `adk_request_confirmation` is present.
    approval_info = check_for_approval(events)

    # -----------------------------------------------------------------------------------------------
    # -----------------------------------------------------------------------------------------------
    # STEP 3: If the event is present, it's a large order - HANDLE APPROVAL WORKFLOW
    if approval_info:
        print(f"⏸️  Pausing for approval...")
        print(f"🤔 Human Decision: {'APPROVE ✅' if auto_approve else 'REJECT ❌'}\n")

        # PATH A: Resume the agent by calling run_async() again with the approval decision
        async for event in shipping_runner.run_async(
            user_id="test_user",
            session_id=session_id,
            new_message=create_approval_response(
                approval_info, auto_approve
            ),  # Send human decision here
            invocation_id=approval_info[
                "invocation_id"
            ],  # Critical: same invocation_id tells ADK to RESUME
        ):
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        print(f"Agent > {part.text}")

    # -----------------------------------------------------------------------------------------------
    # -----------------------------------------------------------------------------------------------
    else:
        # PATH B: If the `adk_request_confirmation` is not present - no approval needed - order completed immediately.
        print_agent_response(events)

    print(f"{'='*60}\n")


print("✅ Workflow function ready")

✅ Workflow function ready


#### **代码拆解**

**步骤 1：向 Agent 发送初始请求**
- 调用 `run_async()` 启动 Agent 执行
- 收集全部事件到列表便于后续检查

**步骤 2：检测暂停**
- 调用 `check_for_approval(events)` 查找特殊事件 `adk_request_confirmation`
- 若存在，返回审批信息（包含 `invocation_id`）；若不存在，表示已直接完成

**步骤 3：恢复执行**

PATH A：
- 若有审批信息，表示 Agent 此时已暂停等待人工输入
- 拿到人工输入后，再次调用 `run_async()` 并传入该输入
- **关键：** 使用同一个 `invocation_id`（告诉 ADK 是恢复而非重启）
- 输出恢复后的最终响应

PATH B：
- 若没有审批信息，说明无需审批，Agent 已直接完成执行。

### 🎬 4.5：演示：测试工作流

现在运行演示。你会看到调用逻辑更清晰了：暂停/恢复相关复杂细节都被封装在 `run_workflow` 辅助函数中，我们可以专注于 Agent 要完成的任务本身。

**注意：** 你可能会看到类似 `Warning: there are non-text parts in the response: ['function_call']` 的提示，这是正常现象，可忽略。它只表示 Agent 在生成文本之外还调用了工具。

In [19]:
# Demo 1: It's a small order. Agent receives auto-approved status from tool
await run_shipping_workflow("Ship 3 containers to Singapore")

# Demo 2: Workflow simulates human decision: APPROVE ✅
await run_shipping_workflow("Ship 10 containers to Rotterdam", auto_approve=True)

# Demo 3: Workflow simulates human decision: REJECT ❌
await run_shipping_workflow("Ship 8 containers to Los Angeles", auto_approve=False)


User > Ship 3 containers to Singapore



Agent > I have placed an order for 3 containers to Singapore. The order has been approved and the order ID is ORD-3-AUTO.


User > Ship 10 containers to Rotterdam



/usr/local/lib/python3.11/dist-packages/google/adk/tools/tool_context.py:92: UserWarning: [EXPERIMENTAL] ToolConfirmation: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  ToolConfirmation(
/usr/local/lib/python3.11/dist-packages/google/adk/agents/invocation_context.py:298: UserWarning: [EXPERIMENTAL] BaseAgentState: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self.agent_states[event.author] = BaseAgentState()


⏸️  Pausing for approval...
🤔 Human Decision: APPROVE ✅

Agent > Shipping order approved. Order ID: ORD-10-HUMAN. 10 containers will be shipped to Rotterdam.


User > Ship 8 containers to Los Angeles



⏸️  Pausing for approval...
🤔 Human Decision: REJECT ❌

Agent > Shipping order rejected: 8 containers to Los Angeles.



### 4.6：（可选）完整执行流程

下面给出一条完整工作流追踪示例。

**TL;DR：** 工具在 TIME 6 暂停，工作流在 TIME 8 发现，TIME 10 使用同一个 `invocation_id="abc123"` 恢复。

**详细时间线：**

当你运行 `run_shipping_workflow("Ship 10 containers to Rotterdam", auto_approve=True)` 时，会发生：

```
TIME 1: User sends "Ship 10 containers to Rotterdam"
        ↓
TIME 2: Workflow calls shipping_runner.run_async(...)
        ADK assigns a unique invocation_id = "abc123"
        ↓
TIME 3: Agent receives user message, decides to use place_shipping_order tool
        ↓
TIME 4: ADK calls place_shipping_order(10, "Rotterdam", tool_context)
        ↓
TIME 5: Tool checks: num_containers (10) > 5
        Tool calls tool_context.request_confirmation(...)
        ↓
TIME 6: Tool returns {'status': 'pending', ...}
        ↓
TIME 7: ADK creates adk_request_confirmation event with invocation_id="abc123"
        ↓
TIME 8: Workflow detects the event via check_for_approval()
        Saves approval_id and invocation_id="abc123"
        ↓
TIME 9: Workflow gets human decision → True (approve)
        ↓
TIME 10: Workflow calls shipping_runner.run_async(..., invocation_id="abc123")
         Passes approval decision as FunctionResponse
         ↓
TIME 11: ADK sees invocation_id="abc123" - knows to RESUME (instead of starting new)
         Loads saved state from TIME 7
         ↓
TIME 12: ADK calls place_shipping_order again with same parameters
         But now tool_context.tool_confirmation.confirmed = True
         ↓
TIME 13: Tool returns {'status': 'approved', 'order_id': 'ORD-10-HUMAN', ...}
         ↓
TIME 14: Agent receives result and responds to user
```

**关键点：** `invocation_id` 是 ADK 区分“恢复已暂停执行”与“新开一次执行”的核心标识。

---

## 📊 第 5 部分：总结 - 高级工具的关键模式

在本 Notebook 中，你实现了两种面向生产的强大模式，用于把 Agent 能力扩展到“简单函数调用”之外。

| 模式 | 适用场景 | 关键 ADK 组件 |
| :--- | :--- | :--- |
| **MCP Integration** | 你需要连接**外部标准化服务**（如时间、数据库、文件系统），且不想手写集成代码。 | `McpToolset` |
| **Long-Running Operations** | 你需要**暂停工作流**等待外部事件，典型是**人审（human-in-the-loop）**、长任务或合规/安全检查。 | `ToolContext`, `request_confirmation`, `App`, `ResumabilityConfig` |

### 🚀 面向生产的核心能力

你现在已经理解如何构建这样的 Agent：
- 🌐 **可扩展**：复用社区工具，而不是事事自建
- ⏳ **可跨时执行**：处理跨分钟、小时甚至天级别的任务  
- 🔒 **可合规治理**：在关键操作中加入人工监督
- 🔄 **可状态恢复**：在中断后从原位置继续

**建议路线：** 先从 custom tools 开始 -> 再接入 MCP services -> 按需加入 long-running 机制

## 🎯 练习：构建一个带“成本审批”的图像生成 Agent

**场景：**

构建一个通过 MCP server 生成图片的 Agent，但对“批量生成”要求审批：
- 单图请求（1 张）：自动通过，立即生成
- 批量请求（>1 张）：先暂停并请求审批，再执行多图生成
- 尝试不同的公开可用 Image Generation MCP Servers

---

## 🎉 恭喜！你已掌握 Agent 模式与最佳实践

你已经学会如何构建能够处理真实复杂流程的 Agent：既能接入外部系统，也能跨时间维度持续执行。

**ℹ️ 说明：无需提交！**

这个 Notebook 仅用于动手实践与学习。完成课程**不需要**把它提交到任何地方。

### 📚 继续学习

- [ADK Documentation](https://google.github.io/adk-docs/)
- [MCP Tools Documentation](https://google.github.io/adk-docs/tools/mcp-tools/)
- [Long-Running Operations Guide](https://google.github.io/adk-docs/tools/function-tools/)
- [Model Context Protocol Specification](https://spec.modelcontextprotocol.io/)
- [The `App` and `Runner`](https://google.github.io/adk-docs/runtime/)

### 🎯 下一步

你已经打下了生产级 Agent 系统的基础。准备迎接下一个挑战了吗？

继续进入 **Day 3**，学习 **State 与 Memory 管理**！